# Wind Statistics

### Step 1. Import the necessary libraries

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_ROOT = Path(os.environ.get("PANDAS_DATA_ROOT", r"../pandas_exercises"))

def data_file(name):
    direct = DATA_ROOT / name
    if direct.is_file():
        return direct
    matches = list(DATA_ROOT.rglob(name)) if DATA_ROOT.exists() else []
    if len(matches) != 1:
        raise FileNotFoundError(f"Set DATA_ROOT to your extracted data folder. Expected one {name}; found {len(matches)}.")
    return matches[0]


### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/06_Stats/Wind_Stats/wind.data)

### Step 3. Assign it to a variable called data and replace the first 3 columns by a proper datetime index.

In [ ]:
raw = pd.read_csv(data_file('wind.data'), sep=r'\s+')
dates = pd.to_datetime(dict(year=raw['Yr'] + 2000, month=raw['Mo'], day=raw['Dy']))
data = raw.drop(columns=['Yr', 'Mo', 'Dy']).set_axis(pd.DatetimeIndex(dates, name='Date'))
data.head()

### Step 4. Year 2061? Do we really have data from this year? Create a function to fix it and apply it.

In [ ]:
def fix_year(index):
    return pd.to_datetime(dict(year=index.year - np.where(index.year > 2000, 100, 0), month=index.month, day=index.day))

data.index = pd.DatetimeIndex(fix_year(data.index), name='Date')

### Step 5. Set the right dates as the index. Pay attention at the data type, it should be datetime64[ns].

In [ ]:
data.index = pd.DatetimeIndex(data.index).astype('datetime64[ns]')
data = data.sort_index()
data.index

### Step 6. Compute how many values are missing for each location over the entire record.  
#### They should be ignored in all calculations below. 

In [ ]:
data.isna().sum()

### Step 7. Compute how many non-missing values there are in total.

In [ ]:
data.count().sum()

### Step 8. Calculate the mean windspeeds of the windspeeds over all the locations and all the times.
#### A single number for the entire dataset.

In [ ]:
data.sum().sum() / data.count().sum()

### Step 9. Create a DataFrame called loc_stats and calculate the min, max and mean windspeeds and standard deviations of the windspeeds at each location over all the days 

#### A different set of numbers for each location.

In [ ]:
loc_stats = data.agg(['min', 'max', 'mean', 'std']).T
loc_stats

### Step 10. Create a DataFrame called day_stats and calculate the min, max and mean windspeed and standard deviations of the windspeeds across all the locations at each day.

#### A different set of numbers for each day.

In [ ]:
day_stats = pd.DataFrame({'min': data.min(axis=1), 'max': data.max(axis=1), 'mean': data.mean(axis=1), 'std': data.std(axis=1)})
day_stats

### Step 11. Find the average windspeed in January for each location.  
#### Treat January 1961 and January 1962 both as January.

In [ ]:
data.loc[data.index.month == 1].mean()

### Step 12. Downsample the record to a yearly frequency for each location.

In [ ]:
data.resample('YS').mean()

### Step 13. Downsample the record to a monthly frequency for each location.

In [ ]:
data.resample('MS').mean()

### Step 14. Downsample the record to a weekly frequency for each location.

In [ ]:
data.resample('W-SUN').mean()

### Step 15. Calculate the min, max and mean windspeeds and standard deviations of the windspeeds across all locations for each week (assume that the first week starts on January 2 1961) for the first 52 weeks.

In [ ]:
weekly_locations = data.loc['1961-01-02':].resample('W-SUN').agg(['min', 'max', 'mean', 'std']).head(52)
weekly_all = data.loc['1961-01-02':].rename_axis('Date').reset_index().melt(id_vars='Date', value_name='speed').groupby(pd.Grouper(key='Date', freq='W-SUN'))['speed'].agg(['min', 'max', 'mean', 'std']).head(52)
display(weekly_locations)
weekly_all